(cap-fundamentos-grafos)=
# Fundamentos de Grafos e Redes Espaciais

:::{admonition} Objetivos de Aprendizagem
:class: important

Ao final deste capítulo, você será capaz de:

1. **Definir** formalmente grafos, dígrafos, grafos ponderados e multigrafos, reconhecendo a notação $G = (V, E)$ e suas variantes.
2. **Calcular e interpretar** grau, grau de entrada e de saída, vizinhança e densidade, relacionando-os por meio do *lema do aperto de mãos*.
3. **Distinguir** as representações computacionais de um grafo (matriz e lista de adjacência) e justificar por que redes espaciais são tipicamente esparsas.
4. **Caracterizar** um *grafo espacial*, diferenciando distância topológica, distância métrica sobre a rede e distância euclidiana ou geodésica.
5. **Formalizar** o problema do caminho mínimo e descrever o algoritmo de Dijkstra.
6. **Construir, consultar e visualizar** grafos, dígrafos e multigrafos com a biblioteca NetworkX, usando coordenadas geográficas reais para posicionar os nós.
:::

A análise de redes espaciais investiga como elementos distribuídos no espaço geográfico se organizam e se conectam: interseções e ruas de uma cidade, estações e linhas de um sistema de transporte, reservatórios e adutoras de uma rede de abastecimento, fragmentos de habitat e corredores ecológicos. Em todos esses casos, o objeto de interesse não é apenas *onde* cada elemento está, mas *como* os elementos se relacionam. A linguagem matemática que torna essas relações explícitas e computáveis é a **teoria dos grafos**.

A origem da teoria remonta a 1736, quando Leonhard Euler demonstrou ser impossível atravessar as sete pontes de Königsberg passando por cada uma exatamente uma vez {cite:p}`euler1736`. O argumento de Euler é, em essência, espacial: ele abstraiu margens e ilhas como pontos e pontes como ligações, descartando a geometria e preservando apenas a *topologia* do problema. Essa abstração, que ignora formas e distâncias para reter conexões, é o ponto de partida deste livro. Veremos, porém, que em redes espaciais a geometria retorna com força: comprimentos, coordenadas e orientações passam a ser atributos essenciais do grafo {cite:p}`barthelemy2011`.

Este capítulo estabelece o vocabulário e a notação usados em todo o livro. Na primeira parte, apresentamos as definições formais; na segunda, construímos três estudos guiados com a biblioteca NetworkX {cite:p}`hagberg2008`, envolvendo capitais brasileiras conectadas por rodovias.

## Grafos: definições fundamentais

Um **grafo** (não direcionado) é um par ordenado

$$
G = (V, E), \qquad E \subseteq \big\{ \{u, v\} : u, v \in V \big\},
$$ (eq-grafo)

em que $V$ é um conjunto finito de **vértices** (ou **nós**) e $E$ é um conjunto de **arestas** (ou **ligações**), cada uma formada por um par não ordenado de vértices. O número de vértices $n = |V|$ é a **ordem** do grafo, e o número de arestas $m = |E|$ é o seu **tamanho**.

Quando $e = \{u, v\} \in E$, dizemos que $u$ e $v$ são **adjacentes** (ou vizinhos) e que $e$ é **incidente** a ambos. A **vizinhança** de um vértice $v$ é o conjunto

$$
N(v) = \{\, u \in V : \{u, v\} \in E \,\}.
$$

### Grau de um vértice

O **grau** de $v$, denotado $\deg(v)$, é o número de arestas incidentes a $v$; em um grafo simples, $\deg(v) = |N(v)|$. Como cada aresta contribui com uma unidade para o grau de cada uma de suas extremidades, vale o resultado a seguir, conhecido como **lema do aperto de mãos**:

$$
\sum_{v \in V} \deg(v) = 2m .
$$ (eq-aperto-maos)

Uma consequência imediata de {eq}`eq-aperto-maos` é que o **grau médio** de um grafo é

$$
\bar{k} = \frac{1}{n} \sum_{v \in V} \deg(v) = \frac{2m}{n}.
$$ (eq-grau-medio)

Em uma rede viária, $\deg(v)$ aproxima o número de segmentos de rua que convergem em uma interseção, e $\bar{k}$ é um primeiro indicador da conectividade da malha. Retomaremos essas grandezas no {ref}`cap-metricas-centralidade`.

### Tipologia de grafos

A definição {eq}`eq-grafo` é o caso mais simples. Na prática, os problemas espaciais exigem variações que acrescentam direção, pesos ou múltiplas ligações entre os mesmos vértices.

**Grafos não direcionados.** As arestas não têm sentido: a relação entre $u$ e $v$ é simétrica. Uma malha de estradas de mão dupla entre cidades é o exemplo típico ({numref}`fig-grafo-nao-direcionado`).

```{figure} ../images/figura1.png
:name: fig-grafo-nao-direcionado
:width: 360px
:align: center

Grafo não direcionado: as ligações são simétricas e podem ser percorridas em ambos os sentidos.
```

**Grafos direcionados (dígrafos).** Em um dígrafo $D = (V, A)$, o conjunto de **arcos** é formado por pares *ordenados*, $A \subseteq V \times V$. O arco $(u, v)$ parte de $u$ (sua cauda) e chega em $v$ (sua cabeça). Ruas de mão única e rotas aéreas são modeladas dessa forma ({numref}`fig-digrafo`). Cada vértice passa a ter um **grau de entrada** $\deg^{-}(v)$ e um **grau de saída** $\deg^{+}(v)$, e o lema do aperto de mãos assume a forma

$$
\sum_{v \in V} \deg^{-}(v) \;=\; \sum_{v \in V} \deg^{+}(v) \;=\; m .
$$ (eq-aperto-maos-dir)

```{figure} ../images/figura2.png
:name: fig-digrafo
:width: 360px
:align: center

Dígrafo: cada arco possui um sentido, da cauda para a cabeça.
```

**Grafos ponderados.** Associa-se a cada aresta um **peso** por meio de uma função $w : E \to \mathbb{R}_{\geq 0}$. Em redes espaciais, o peso costuma representar comprimento (em metros), tempo de viagem (em segundos) ou custo generalizado ({numref}`fig-grafo-ponderado`). Quando não há pesos explícitos, dizemos que o grafo é **não ponderado**, o que equivale a adotar $w(e) = 1$ para toda aresta.

```{figure} ../images/figura3.png
:name: fig-grafo-ponderado
:width: 360px
:align: center

Grafo ponderado: cada aresta carrega um valor numérico, como distância ou tempo.
```

**Grafos simples e multigrafos.** Um grafo é **simples** quando não possui **laços** (arestas do tipo $\{v, v\}$) nem **arestas paralelas** (duas ou mais arestas ligando o mesmo par de vértices). Um **multigrafo** admite arestas paralelas; formalmente, $E$ passa a ser um *multiconjunto*, e cada aresta é identificada por uma tripla $(u, v, k)$, em que $k$ é uma **chave** que distingue as arestas paralelas ({numref}`fig-multigrafo`). Duas cidades ligadas simultaneamente por rodovia, ferrovia e rota aérea formam um multigrafo.

```{figure} ../images/figura4.png
:name: fig-multigrafo
:width: 360px
:align: center

Multigrafo: múltiplas arestas conectam o mesmo par de vértices.
```

**Grafos completos.** O grafo completo $K_n$ possui uma aresta entre cada par de vértices distintos ({numref}`fig-grafo-completo`). Seu tamanho é o máximo possível para um grafo simples de ordem $n$:

$$
m_{\max} = \binom{n}{2} = \frac{n(n-1)}{2}.
$$ (eq-m-max)

```{figure} ../images/figura5.png
:name: fig-grafo-completo
:width: 360px
:align: center

Grafo completo: todos os vértices são mutuamente adjacentes.
```

A razão entre o número de arestas existentes e o máximo possível define a **densidade** do grafo:

$$
\rho = \frac{m}{m_{\max}} = \frac{2m}{n(n-1)} \quad \text{(não direcionado)}, \qquad
\rho = \frac{m}{n(n-1)} \quad \text{(direcionado)}.
$$ (eq-densidade)

A {numref}`tab-tipos-grafos` resume a correspondência entre os tipos de grafo e as classes da biblioteca NetworkX, que usaremos ao longo do livro.

:::{table} Tipos de grafo e classes correspondentes na NetworkX.
:name: tab-tipos-grafos

| Tipo | Direção | Arestas paralelas | Classe NetworkX | Exemplo espacial |
|---|---|---|---|---|
| Grafo simples | Não | Não | `nx.Graph` | Malha rodoviária entre cidades |
| Dígrafo | Sim | Não | `nx.DiGraph` | Rotas aéreas, fluxos origem–destino |
| Multigrafo | Não | Sim | `nx.MultiGraph` | Ligações multimodais entre cidades |
| Multidígrafo | Sim | Sim | `nx.MultiDiGraph` | Redes viárias urbanas (OSMnx) |

:::

:::{note}
O **multidígrafo** é a estrutura que a biblioteca OSMnx utiliza para representar redes viárias. A direção permite modelar ruas de mão única; as arestas paralelas permitem representar, por exemplo, duas vias distintas que ligam as mesmas duas interseções. Estudaremos essa representação em detalhe no {ref}`cap-osmnx`.
:::

## Representações computacionais

Para manipular grafos em um computador, precisamos de uma estrutura de dados. As duas representações clássicas são a matriz de adjacência e a lista de adjacência.

A **matriz de adjacência** de um grafo com vértices $v_1, \dots, v_n$ é a matriz $\mathbf{A} \in \{0,1\}^{n \times n}$ tal que

$$
A_{ij} =
\begin{cases}
1, & \text{se } \{v_i, v_j\} \in E,\\
0, & \text{caso contrário.}
\end{cases}
$$ (eq-matriz-adj)

Em grafos não direcionados, $\mathbf{A}$ é simétrica, e o grau de $v_i$ é a soma da linha $i$: $\deg(v_i) = \sum_j A_{ij}$. Em grafos ponderados, é comum usar a **matriz de pesos** $\mathbf{W}$, com $W_{ij} = w(\{v_i, v_j\})$.

A **lista de adjacência** armazena, para cada vértice, apenas a lista de seus vizinhos. Enquanto a matriz ocupa memória proporcional a $n^2$, a lista ocupa memória proporcional a $n + m$.

:::{tip}
Redes espaciais são **esparsas**: por restrições físicas, uma interseção raramente conecta mais que quatro ou cinco ruas, de modo que $\bar{k}$ permanece pequeno e praticamente independente de $n$. Assim, $m = \bar{k}\,n/2$ cresce linearmente com $n$, e a densidade {eq}`eq-densidade` tende a zero à medida que a rede cresce. Para uma rede com $10^5$ interseções, a matriz de adjacência teria $10^{10}$ células, quase todas nulas. Por isso a NetworkX armazena grafos como **dicionários de dicionários**, uma variante da lista de adjacência.
:::

## Caminhos, conectividade e distâncias

Um **passeio** entre $s$ e $t$ é uma sequência de vértices $P = (v_0 = s, v_1, \dots, v_\ell = t)$ tal que cada par consecutivo é adjacente. Se não houver vértices repetidos, o passeio é um **caminho**. O número de arestas $\ell$ é o **comprimento topológico** do caminho. Em um grafo ponderado, o **custo** do caminho é

$$
c(P) = \sum_{i=0}^{\ell-1} w(v_i, v_{i+1}),
$$ (eq-custo-caminho)

e a **distância na rede** entre $s$ e $t$ é o menor custo entre todos os caminhos que os ligam:

$$
d_G(s, t) = \min_{P \in \mathcal{P}_{st}} c(P),
$$ (eq-distancia-rede)

em que $\mathcal{P}_{st}$ é o conjunto de caminhos de $s$ a $t$ (por convenção, $d_G(s,t) = \infty$ quando esse conjunto é vazio). Um grafo não direcionado é **conexo** se existe caminho entre qualquer par de vértices; seus subgrafos conexos maximais são as **componentes conexas**. Em dígrafos, distinguimos a conectividade **forte** (existe caminho orientado de $u$ para $v$ *e* de $v$ para $u$ para todo par) da conectividade **fraca** (o grafo é conexo quando os sentidos são ignorados).

:::{warning}
Em dígrafos, a distância geralmente **não é simétrica**: $d_G(s,t) \neq d_G(t,s)$. Em uma área central com muitas ruas de mão única, ir de A até B pode exigir um percurso muito diferente do retorno de B até A. Essa assimetria terá consequências diretas no cálculo das centralidades de proximidade no {ref}`cap-metricas-centralidade`.
:::

### O problema do caminho mínimo e o algoritmo de Dijkstra

Encontrar o caminho que realiza {eq}`eq-distancia-rede` é o **problema do caminho mínimo**, central para o roteamento (Parte IV). Para pesos não negativos, a solução clássica é o algoritmo de Dijkstra {cite:p}`dijkstra1959`, que mantém uma estimativa $d(v)$ da distância de $s$ até cada vértice e a refina sucessivamente pela operação de **relaxação**:

$$
d(v) \leftarrow \min\big\{\, d(v),\; d(u) + w(u, v) \,\big\}.
$$ (eq-relaxacao)

:::{admonition} Algoritmo 1.1: Dijkstra
:class: seealso

**Entrada:** grafo $G=(V,E)$, pesos $w \geq 0$, origem $s$.
**Saída:** distâncias $d(v)$ e predecessores $\pi(v)$ para todo $v \in V$.

1. Para todo $v \in V$: $d(v) \leftarrow \infty$, $\pi(v) \leftarrow \varnothing$. Faça $d(s) \leftarrow 0$.
2. Insira todos os vértices em uma fila de prioridade $Q$, ordenada por $d$.
3. Enquanto $Q \neq \varnothing$:
   1. Remova de $Q$ o vértice $u$ com menor $d(u)$; a distância $d(u)$ torna-se **definitiva**.
   2. Para cada vizinho $v$ de $u$: se $d(u) + w(u,v) < d(v)$, faça $d(v) \leftarrow d(u) + w(u,v)$ e $\pi(v) \leftarrow u$.
4. Retorne $d$ e $\pi$. O caminho até $t$ é reconstruído seguindo $\pi$ de $t$ até $s$.
:::

Com uma fila de prioridade implementada por *heap* binário, o algoritmo executa em tempo $O\big((n + m)\log n\big)$. Em grafos não ponderados, basta uma **busca em largura**, que executa em $O(n + m)$. Variantes como o algoritmo A\* {cite:p}`hart1968`, que usa a distância euclidiana até o destino como heurística, exploram a natureza espacial do grafo para acelerar a busca.

## Grafos espaciais

Até aqui, as definições foram puramente topológicas. Um **grafo espacial** {cite:p}`barthelemy2011` é um grafo cujos vértices estão **imersos** em um espaço métrico, em geral o plano $\mathbb{R}^2$ ou a superfície terrestre, por meio de uma função de posição

$$
\mathbf{x} : V \to \mathbb{R}^2, \qquad v \mapsto \mathbf{x}_v = (x_v, y_v),
$$ (eq-imersao)

e cujas arestas geralmente têm custo associado à sua extensão física. Três propriedades distinguem esses grafos das redes "abstratas", como as redes sociais:

1. **Restrição de grau.** O espaço físico limita o número de conexões por vértice, de modo que a distribuição de graus é concentrada em poucos valores (tipicamente de 1 a 5 em redes viárias).
2. **Planaridade aproximada.** Um grafo é **planar** se pode ser desenhado no plano sem cruzamento de arestas. Redes viárias são *quase* planares: viadutos, túneis e passarelas são as exceções.
3. **Custo de ligação dependente da distância.** Conexões longas são caras, e o peso de uma aresta costuma ser proporcional ao seu comprimento.

É fundamental distinguir três noções de distância em redes espaciais:

- a **distância topológica**, isto é, o número de arestas de um caminho mínimo não ponderado;
- a **distância na rede** $d_G(s,t)$, dada por {eq}`eq-distancia-rede` com pesos iguais aos comprimentos das arestas;
- a **distância euclidiana** $d_E(s,t) = \lVert \mathbf{x}_s - \mathbf{x}_t \rVert_2$ ou, em coordenadas geográficas, a **distância geodésica** (de grande círculo). Para pontos de latitude $\varphi$ e longitude $\lambda$ sobre uma esfera de raio $R \approx 6\,371$ km, a fórmula de *haversine* fornece

$$
d_{gc} = 2R \arcsin\!\sqrt{\sin^2\!\Big(\frac{\Delta\varphi}{2}\Big) + \cos\varphi_1 \cos\varphi_2 \sin^2\!\Big(\frac{\Delta\lambda}{2}\Big)} .
$$ (eq-haversine)

A razão entre a distância na rede e a distância em linha reta mede quanto a rede obriga o viajante a desviar e é chamada de **circuidade** (ou *detour index*):

$$
\mathrm{circ}(s,t) = \frac{d_G(s,t)}{d_{gc}(s,t)} \geq 1 .
$$ (eq-circuidade)

:::{note}
**Representação primal e dual.** Na representação **primal**, adotada neste livro e pela OSMnx, interseções são vértices e segmentos de rua são arestas {cite:p}`porta2006`. Na representação **dual**, comum na *sintaxe espacial*, cada rua (ou linha axial) é um vértice, e duas ruas são adjacentes quando se cruzam. A escolha da representação altera o significado de todas as métricas: na dual, o grau mede quantas ruas cruzam uma dada rua, e não quantos segmentos chegam a uma interseção.
:::

## A biblioteca NetworkX

Ferramentas de SIG *desktop* oferecem análises de rede por meio de interfaces gráficas, mas limitam a automação, a reprodutibilidade e a customização de algoritmos. O ecossistema Python permite escrever análises como *scripts* versionáveis, integrá-las a bibliotecas de dados espaciais (GeoPandas, Shapely), de aprendizado de máquina (scikit-learn) e de visualização (Matplotlib, Folium), e executá-las em servidores ou na nuvem.

Nesse ecossistema, a **NetworkX** {cite:p}`hagberg2008` é a biblioteca de referência para a criação, manipulação e análise de grafos. Ela oferece as quatro classes da {numref}`tab-tipos-grafos`, permite associar atributos arbitrários ao grafo, aos nós e às arestas, e implementa centenas de algoritmos: caminhos mínimos, componentes conexas, centralidades, detecção de comunidades, entre outros. Também lê e grava formatos de intercâmbio como GraphML, GML e listas de arestas. Sua visualização é propositalmente simples, construída sobre a Matplotlib; para mapas, usaremos a OSMnx e a GeoPandas a partir do {ref}`cap-osmnx`.

Nas seções seguintes, desenvolvemos três estudos guiados de complexidade crescente: um grafo simples, um dígrafo e um multigrafo, todos envolvendo capitais brasileiras.

In [ ]:
import math

import matplotlib.pyplot as plt
import networkx as nx
import pandas as pd

print("NetworkX", nx.__version__)

## Estudo guiado 1: um grafo não direcionado

Vamos modelar as capitais da Região Sudeste e suas ligações rodoviárias como um grafo não direcionado. Além da estrutura, registraremos **atributos**: a população de cada capital, as coordenadas geográficas (que tornam o grafo *espacial*) e a distância rodoviária de cada ligação.

### Criação do grafo e atributos globais

Um objeto `nx.Graph` vazio é criado pela chamada ao construtor. O dicionário `G.graph` armazena atributos que descrevem o grafo como um todo, úteis para documentar a origem e a versão dos dados.

In [ ]:
G = nx.Graph()

G.graph["projeto"] = "Análise de Capitais Brasileiras"
G.graph["data_criacao"] = "2024-01-02"
G.graph["versao"] = "1.0"
G.graph["crs"] = "EPSG:4326"  # coordenadas em graus (WGS 84)

G.graph

### Inserção de nós e atributos de nós

Nós são inseridos com `add_node`. Qualquer objeto *hashable* pode ser um nó; aqui, usamos o nome da cidade. Atributos podem ser passados como argumentos nomeados. Adotamos as chaves `x` (longitude) e `y` (latitude), a mesma convenção usada pela OSMnx.

In [ ]:
capitais = {
    # nome: (longitude, latitude, população em 2022)
    "São Paulo": (-46.633, -23.550, 11_451_245),
    "Rio de Janeiro": (-43.173, -22.907, 6_211_423),
    "Belo Horizonte": (-43.938, -19.920, 2_315_560),
    "Vitória": (-40.308, -20.320, 322_869),
}

for nome, (lon, lat, pop) in capitais.items():
    G.add_node(nome, x=lon, y=lat, populacao=pop)

G.nodes(data=True)

:::{note}
Os valores populacionais são do Censo Demográfico de 2022 (IBGE), e as coordenadas são aproximadas. O exemplo tem finalidade didática; para análises reais, use sempre fontes oficiais e registre a data de referência nos atributos do grafo.
:::

Como o grafo é espacial, podemos posicionar os nós segundo suas coordenadas, em vez de usar um *layout* abstrato. A função auxiliar abaixo será reutilizada em todo o capítulo.

In [ ]:
def posicoes_geograficas(grafo):
    """Retorna {nó: (x, y)} a partir dos atributos 'x' e 'y' dos nós."""
    return {n: (d["x"], d["y"]) for n, d in grafo.nodes(data=True)}


def desenhar(grafo, titulo, rotulo_aresta=None, caminho=None, cor_caminho="tab:blue",
             ax=None, **kwargs):
    """Desenha o grafo em coordenadas geográficas.

    rotulo_aresta: nome do atributo exibido sobre as arestas (opcional).
    caminho: lista de nós cujo trajeto será destacado (opcional).
    """
    if ax is None:
        _, ax = plt.subplots(figsize=(7, 6))
    pos = posicoes_geograficas(grafo)
    opcoes = dict(node_color="lightblue", node_size=900, font_size=9, font_weight="bold")
    opcoes.update(kwargs)
    nx.draw_networkx(grafo, pos, ax=ax, **opcoes)
    if caminho:
        arestas = list(zip(caminho, caminho[1:]))
        nx.draw_networkx_edges(grafo, pos, edgelist=arestas, edge_color=cor_caminho, width=3,
                               arrowsize=opcoes.get("arrowsize", 10),
                               node_size=opcoes["node_size"], ax=ax)
    if rotulo_aresta:
        rotulos = nx.get_edge_attributes(grafo, rotulo_aresta)
        nx.draw_networkx_edge_labels(grafo, pos, edge_labels=rotulos, ax=ax, font_size=8)
    ax.set_title(titulo)
    ax.set_xlabel("Longitude (°)")
    ax.set_ylabel("Latitude (°)")
    ax.tick_params(left=True, bottom=True, labelleft=True, labelbottom=True)
    ax.set_aspect("equal")
    ax.margins(0.15)
    return ax

In [ ]:
desenhar(G, "Capitais do Sudeste (apenas nós)")
plt.show()

### Remoção e reinserção de nós

A função `remove_node` remove um nó **e todas as arestas incidentes a ele**. Para ilustrar, removemos e reinserimos o Rio de Janeiro.

In [ ]:
G.remove_node("Rio de Janeiro")
print("Após remoção:", list(G.nodes))

G.add_node("Rio de Janeiro", **dict(zip(("x", "y", "populacao"), capitais["Rio de Janeiro"])))
print("Após reinserção:", list(G.nodes))

### Inserção de arestas e atributos de arestas

Arestas são inseridas com `add_edge(u, v, **atributos)`. Se algum dos nós ainda não existir, ele é criado automaticamente. Esse comportamento é conveniente, mas pode introduzir nós indesejados por erros de digitação.

In [ ]:
ligacoes = [
    # (origem, destino, distância rodoviária aproximada em km)
    ("São Paulo", "Rio de Janeiro", 430),
    ("São Paulo", "Belo Horizonte", 590),
    ("Rio de Janeiro", "Belo Horizonte", 440),
    ("Rio de Janeiro", "Vitória", 520),
    ("Belo Horizonte", "Vitória", 530),
]
G.add_weighted_edges_from(ligacoes, weight="distancia")

print(f"n = {G.number_of_nodes()}, m = {G.number_of_edges()}")
G.edges(data=True)

In [ ]:
desenhar(G, "Capitais do Sudeste e ligações rodoviárias", rotulo_aresta="distancia")
plt.show()

Atributos individuais são acessados com a notação `G.edges[u, v][atributo]`. Como o grafo é não direcionado, `G.edges["São Paulo", "Belo Horizonte"]` e `G.edges["Belo Horizonte", "São Paulo"]` referem-se à mesma aresta.

In [ ]:
print("SP–BH:", G.edges["São Paulo", "Belo Horizonte"]["distancia"], "km")
print("BH–SP:", G.edges["Belo Horizonte", "São Paulo"]["distancia"], "km")

Para remover uma aresta, sem remover seus nós, usa-se `remove_edge`:

In [ ]:
G.remove_edge("Belo Horizonte", "Vitória")
print("Aresta BH–Vitória presente?", G.has_edge("Belo Horizonte", "Vitória"))

G.add_edge("Belo Horizonte", "Vitória", distancia=530)
print("Aresta BH–Vitória presente?", G.has_edge("Belo Horizonte", "Vitória"))

Por fim, incluímos Curitiba, ligada a São Paulo. Note que o grafo passa a extrapolar a Região Sudeste, o que atualizamos no atributo global `projeto`.

In [ ]:
G.add_node("Curitiba", x=-49.273, y=-25.428, populacao=1_773_718)
G.add_edge("Curitiba", "São Paulo", distancia=408)
G.graph["projeto"] = "Capitais do Sudeste e Curitiba"

### Grau, vizinhança e verificação do lema do aperto de mãos

O atributo `G.degree` fornece o grau de cada nó, e `G.adj[v]` (ou `G[v]`) fornece a vizinhança $N(v)$ com os atributos das arestas correspondentes.

In [ ]:
graus = dict(G.degree)
print("Graus:", graus)
print("Vizinhos de São Paulo:", list(G.adj["São Paulo"]))
print("Vizinhos de Curitiba:", list(G.adj["Curitiba"]))

Podemos verificar numericamente o lema {eq}`eq-aperto-maos` e calcular o grau médio {eq}`eq-grau-medio` e a densidade {eq}`eq-densidade`:

In [ ]:
n, m = G.number_of_nodes(), G.number_of_edges()
soma_graus = sum(graus.values())

print(f"Soma dos graus = {soma_graus} ; 2m = {2 * m}")
print(f"Grau médio  k̄ = 2m/n = {2 * m / n:.2f}")
print(f"Densidade   ρ = {nx.density(G):.3f}  (fórmula: {2 * m / (n * (n - 1)):.3f})")

### Matriz de adjacência e matriz de pesos

A função `nx.to_pandas_adjacency` produz a matriz {eq}`eq-matriz-adj` como um `DataFrame`. Com `weight=None`, obtemos a matriz binária; com `weight="distancia"`, a matriz de pesos $\mathbf{W}$.

In [ ]:
A = nx.to_pandas_adjacency(G, weight=None, dtype=int)
A

In [ ]:
W = nx.to_pandas_adjacency(G, weight="distancia", dtype=int)
W

Observe que as somas das linhas de `A` reproduzem os graus, que ambas as matrizes são simétricas e que a maioria das células é nula mesmo em um grafo tão pequeno.

### Da topologia à geometria: circuidade entre capitais

Como os nós carregam coordenadas, podemos comparar a distância rodoviária com a distância geodésica {eq}`eq-haversine` e calcular a circuidade {eq}`eq-circuidade` de cada ligação.

In [ ]:
def haversine_km(lon1, lat1, lon2, lat2, raio=6371.0):
    """Distância de grande círculo (km) entre dois pontos em graus decimais."""
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = phi2 - phi1
    dlmb = math.radians(lon2 - lon1)
    a = math.sin(dphi / 2) ** 2 + math.cos(phi1) * math.cos(phi2) * math.sin(dlmb / 2) ** 2
    return 2 * raio * math.asin(math.sqrt(a))


linhas = []
for u, v, d in G.edges(data=True):
    d_gc = haversine_km(G.nodes[u]["x"], G.nodes[u]["y"], G.nodes[v]["x"], G.nodes[v]["y"])
    linhas.append({"ligação": f"{u} – {v}", "rodovia_km": d["distancia"],
                   "geodésica_km": round(d_gc), "circuidade": round(d["distancia"] / d_gc, 2)})

pd.DataFrame(linhas).sort_values("circuidade", ascending=False)

:::{tip}
Circuidades próximas de 1 indicam ligações praticamente retilíneas; valores elevados revelam barreiras físicas, como serras, baías e áreas protegidas, que obrigam a rodovia a desviar. No {ref}`cap-metricas-centralidade`, a mesma ideia será aplicada a milhares de segmentos de uma rede urbana por meio da métrica `circuity_avg` da OSMnx.
:::

## Estudo guiado 2: um dígrafo de viagens

Agora modelamos um **itinerário de viagem** de Curitiba a Vitória. Como as viagens têm sentido, usamos um dígrafo (`nx.DiGraph`). O objetivo é explorar graus de entrada e saída, sucessores e predecessores e, sobretudo, o problema do caminho mínimo.

### Construção do dígrafo

Reaproveitamos os atributos dos nós do grafo `G` e inserimos os arcos com `add_edge(u, v)`, que agora representa o arco **de** `u` **para** `v`.

In [ ]:
D_viagem = nx.DiGraph()
D_viagem.add_nodes_from(G.nodes(data=True))

D_viagem.add_edge("Curitiba", "São Paulo", distancia=408)
D_viagem.add_edge("São Paulo", "Rio de Janeiro", distancia=429)
D_viagem.add_edge("Rio de Janeiro", "Vitória", distancia=521)
# Ramificação alternativa via Belo Horizonte
D_viagem.add_edge("São Paulo", "Belo Horizonte", distancia=586)
D_viagem.add_edge("Belo Horizonte", "Vitória", distancia=524)

D_viagem.edges(data=True)

In [ ]:
desenhar(D_viagem, "Itinerários de Curitiba a Vitória", rotulo_aresta="distancia",
         arrows=True, arrowsize=18, edge_color="gray", width=1.5)
plt.show()

### Graus de entrada e de saída

Em um dígrafo, `degree` retorna a soma $\deg^{-}(v) + \deg^{+}(v)$, enquanto `in_degree` e `out_degree` retornam cada componente. A tabela abaixo também verifica a identidade {eq}`eq-aperto-maos-dir`.

In [ ]:
graus_dir = pd.DataFrame({
    "entrada": dict(D_viagem.in_degree),
    "saída": dict(D_viagem.out_degree),
    "total": dict(D_viagem.degree),
})
print(graus_dir)
print("\nΣ entrada =", graus_dir["entrada"].sum(), "| Σ saída =", graus_dir["saída"].sum(),
      "| m =", D_viagem.number_of_edges())

A leitura da tabela é direta: Curitiba é uma **fonte** ($\deg^{-} = 0$), pois o itinerário apenas parte dela; Vitória é um **sumidouro** ($\deg^{+} = 0$); e São Paulo, com $\deg^{+} = 2$, é o ponto de **bifurcação** do itinerário.

### Sucessores e predecessores

Os **sucessores** de $v$ são os vértices alcançáveis por um arco que parte de $v$; os **predecessores** são as caudas dos arcos que chegam a $v$. Em um dígrafo, `D.adj[v]` coincide com os sucessores.

In [ ]:
for cidade in D_viagem.nodes:
    print(f"{cidade:15s} | sucessores: {list(D_viagem.successors(cidade))!s:40s}"
          f"| predecessores: {list(D_viagem.predecessors(cidade))}")

### Caminho mínimo por distância

A função `nx.shortest_path` resolve o problema do caminho mínimo. Quando recebe o argumento `weight`, a NetworkX usa o algoritmo de Dijkstra (Algoritmo 1.1); sem pesos, usa busca em largura.

In [ ]:
caminho_mais_curto = nx.shortest_path(D_viagem, "Curitiba", "Vitória", weight="distancia")
distancia_total = nx.shortest_path_length(D_viagem, "Curitiba", "Vitória", weight="distancia")

print("Caminho mínimo:", " → ".join(caminho_mais_curto))
print(f"Distância total: {distancia_total} km")

É instrutivo comparar o resultado com **todos** os caminhos simples entre origem e destino, verificando que o algoritmo de fato escolheu o de menor custo {eq}`eq-custo-caminho`:

In [ ]:
for caminho in nx.all_simple_paths(D_viagem, "Curitiba", "Vitória"):
    custo = nx.path_weight(D_viagem, caminho, weight="distancia")
    print(f"{custo:5d} km  " + " → ".join(caminho))

In [ ]:
desenhar(D_viagem, "Caminho mínimo por distância", rotulo_aresta="distancia",
         caminho=caminho_mais_curto, cor_caminho="tab:blue",
         arrows=True, arrowsize=18, edge_color="lightgray", width=1.5)
plt.show()

### Caminho mínimo com custo nos nós: minimizando a população atravessada

Suponha agora que o objetivo seja minimizar a **população total das cidades atravessadas**, por exemplo, para reduzir a exposição de uma carga perigosa. O custo passa a estar nos **nós**, e não nas arestas:

$$
c_{\text{pop}}(P) = \sum_{v \in P} p(v),
$$ (eq-custo-nos)

em que $p(v)$ é a população de $v$. Algoritmos de caminho mínimo operam sobre pesos de arestas, então precisamos **transformar** o custo nodal em custo de arco. Uma transformação exata é atribuir a cada arco $(u, v)$ o custo do nó de chegada:

$$
w'(u, v) = p(v) \quad\Longrightarrow\quad c_{\text{pop}}(P) = p(s) + \sum_{(u,v) \in P} w'(u, v).
$$ (eq-transformacao-no-arco)

Como $p(s)$ é constante para uma origem fixa, minimizar a soma dos pesos $w'$ equivale a minimizar {eq}`eq-custo-nos`. A NetworkX aceita uma **função** como argumento `weight`, com assinatura `f(u, v, atributos_da_aresta)`.

In [ ]:
def peso_populacao(u, v, atributos):
    """Custo do arco (u, v) = população do nó de chegada v."""
    return D_viagem.nodes[v]["populacao"]


caminho_menor_populacao = nx.shortest_path(D_viagem, "Curitiba", "Vitória", weight=peso_populacao)
populacao_total = sum(D_viagem.nodes[c]["populacao"] for c in caminho_menor_populacao)

print("Caminho de menor população:", " → ".join(caminho_menor_populacao))
print(f"População total atravessada: {populacao_total:,}".replace(",", "."))

:::{note}
Uma formulação alternativa comum é $w(u,v) = p(u) + p(v)$. Ela produz o **mesmo caminho ótimo**, pois cada nó intermediário é contado duas vezes e os extremos, uma vez, e esses extremos são fixos. Porém, o custo resultante não corresponde à população atravessada. A transformação {eq}`eq-transformacao-no-arco` é preferível porque o valor minimizado tem interpretação direta.
:::

In [ ]:
desenhar(D_viagem, "Caminho de menor população atravessada", rotulo_aresta="distancia",
         caminho=caminho_menor_populacao, cor_caminho="tab:red",
         arrows=True, arrowsize=18, edge_color="lightgray", width=1.5)
plt.show()

A comparação entre a {numref}`fig-caminho-minimo-distancia` e a {numref}`fig-caminho-minimo-populacao` ilustra um princípio que atravessa todo o livro: **o caminho "ótimo" depende da função de custo**. Distância, tempo, exposição, declividade e custo monetário produzem, em geral, rotas diferentes sobre a mesma rede.

## Estudo guiado 3: um multigrafo multimodal

Belo Horizonte e Vitória são ligadas por rodovia, ferrovia (Estrada de Ferro Vitória a Minas) e rota aérea. Um grafo simples não comporta três arestas entre o mesmo par de nós, mas um **multigrafo** sim. Cada aresta é identificada pela tripla $(u, v, k)$, e escolhemos chaves (`key`) descritivas.

In [ ]:
G_multi = nx.MultiGraph()
for cidade in ("Belo Horizonte", "Vitória"):
    G_multi.add_node(cidade, **G.nodes[cidade])

G_multi.add_edge("Belo Horizonte", "Vitória", key="rodovia", modo="rodovia", distancia=524)
G_multi.add_edge("Belo Horizonte", "Vitória", key="ferrovia", modo="ferrovia", distancia=664)
G_multi.add_edge("Belo Horizonte", "Vitória", key="aereo", modo="aéreo", distancia=380)

for u, v, k, d in G_multi.edges(keys=True, data=True):
    print(f"{u} – {v} [{k:8s}] {d['distancia']:4d} km")

print("\nNúmero de arestas:", G_multi.number_of_edges())
print("Grau de Belo Horizonte:", G_multi.degree["Belo Horizonte"])

Note que o grau de Belo Horizonte é 3, embora ele tenha um único vizinho: em multigrafos, $\deg(v) \neq |N(v)|$. Para visualizar arestas paralelas sem sobreposição, desenhamos cada uma como um arco com curvatura diferente.

In [ ]:
pos = posicoes_geograficas(G_multi)
fig, ax = plt.subplots(figsize=(7, 4))
nx.draw_networkx_nodes(G_multi, pos, node_color="lightblue", node_size=900, ax=ax)
nx.draw_networkx_labels(G_multi, pos, font_size=9, font_weight="bold", ax=ax)

estilos = {"rodovia": ("tab:gray", 0.0), "ferrovia": ("tab:brown", 0.25), "aereo": ("tab:blue", -0.25)}
for chave, (cor, curvatura) in estilos.items():
    nx.draw_networkx_edges(G_multi, pos, edgelist=[("Belo Horizonte", "Vitória", chave)],
                           edge_color=cor, width=2, arrows=True, arrowstyle="-",
                           connectionstyle=f"arc3,rad={curvatura}", ax=ax)

legenda = [plt.Line2D([0], [0], color=c, lw=2, label=k) for k, (c, _) in estilos.items()]
ax.legend(handles=legenda, loc="lower left")
ax.set_title("Ligações multimodais entre Belo Horizonte e Vitória")
ax.margins(0.2)
plt.show()

:::{note}
Na NetworkX, o parâmetro `connectionstyle` só tem efeito quando as arestas são desenhadas como `FancyArrowPatch`, o que exige `arrows=True`. Usamos `arrowstyle="-"` para desenhar arcos sem ponta de seta, preservando a natureza não direcionada do multigrafo.
:::

Ao calcular caminhos mínimos em multigrafos, a NetworkX considera, para cada par de nós adjacentes, **a aresta paralela de menor peso**. Se quisermos obter um grafo simples equivalente para análise, devemos agregar explicitamente as arestas paralelas, por exemplo, mantendo a de menor peso:

In [ ]:
print("Distância mínima BH–Vitória:",
      nx.shortest_path_length(G_multi, "Belo Horizonte", "Vitória", weight="distancia"), "km")

G_simples = nx.Graph()
for u, v, d in G_multi.edges(data=True):
    if not G_simples.has_edge(u, v) or d["distancia"] < G_simples[u][v]["distancia"]:
        G_simples.add_edge(u, v, **d)

G_simples.edges(data=True)

:::{warning}
A conversão direta `nx.Graph(G_multi)` **não** escolhe a aresta de menor peso: ela mantém os atributos da última aresta paralela processada, descartando silenciosamente as demais. Em redes viárias, isso pode trocar um segmento curto por um longo e distorcer todas as distâncias. Sempre agregue arestas paralelas com um critério explícito. A OSMnx oferece `ox.convert.to_digraph`, que mantém a aresta de menor `length`.
:::

## Síntese do capítulo

- Um grafo $G=(V,E)$ abstrai entidades (vértices) e relações (arestas). Direção, pesos e arestas paralelas definem dígrafos, grafos ponderados e multigrafos, com classes correspondentes na NetworkX.
- O **grau** e o **lema do aperto de mãos** conectam propriedades locais ($\deg(v)$) e globais ($m$, $\bar{k}$, $\rho$) do grafo.
- Redes espaciais são **esparsas**, **quase planares** e têm vértices **imersos** no espaço. Por isso distinguimos distância topológica, distância na rede e distância euclidiana ou geodésica, cuja razão define a **circuidade**.
- O **caminho mínimo** depende da função de custo. O algoritmo de Dijkstra resolve o problema para pesos não negativos, e custos nodais podem ser transformados em custos de arco.
- Multigrafos exigem cuidado ao serem convertidos em grafos simples.

## Exercícios Práticos e Desafios

### Exercícios conceituais

1. Um grafo simples não direcionado possui 12 vértices, cada um com grau 3. Quantas arestas ele possui? Existe um grafo simples com 7 vértices, todos de grau 3? Justifique usando {eq}`eq-aperto-maos`.
2. Mostre que, em um dígrafo simples, $0 \leq \rho \leq 1$ com $\rho$ definido por {eq}`eq-densidade`. O que ocorre com esse limite superior em um multidígrafo?
3. Explique por que a densidade de uma rede viária tende a zero à medida que a área de estudo aumenta, enquanto o grau médio permanece aproximadamente constante.
4. Dê um exemplo concreto, em uma cidade que você conheça, em que $d_G(A,B) \neq d_G(B,A)$.

### Exercícios computacionais

5. Estenda o grafo `G` com as capitais da Região Sul (Florianópolis e Porto Alegre), incluindo coordenadas, população e ligações rodoviárias plausíveis. Recalcule $\bar{k}$, $\rho$ e a tabela de circuidades.
6. Usando `nx.single_source_dijkstra`, obtenha as distâncias de São Paulo para todas as demais capitais do grafo estendido. Compare os resultados com as distâncias geodésicas e identifique o destino de maior circuidade.
7. Implemente o Algoritmo 1.1 em Python puro, usando o módulo `heapq`, e valide sua implementação comparando os resultados com `nx.single_source_dijkstra_path_length` no dígrafo `D_viagem`.
8. Adicione ao multigrafo `G_multi` um atributo `tempo_h` (tempo de viagem em horas) para cada modo e determine o modo ótimo segundo (a) distância e (b) tempo. Em seguida, construa um grafo simples que preserve, para cada par de nós, a aresta de menor tempo.

### Desafios

9. **Custo generalizado.** Defina, para cada arco do dígrafo `D_viagem`, um custo generalizado $c(u,v) = \alpha \cdot \text{distância} + \beta \cdot p(v)$. Varie a razão $\beta/\alpha$ e determine o valor a partir do qual o caminho ótimo muda. Interprete o resultado.
10. **Planaridade.** Use `nx.check_planarity` para verificar se $K_4$ e $K_5$ são planares. Em seguida, construa um pequeno grafo espacial representando um viaduto sobre uma avenida e discuta se ele deveria ser modelado como planar.

:::{admonition} Dica para o Exercício 7
:class: dropdown

Mantenha um dicionário `dist` inicializado com `math.inf` e uma fila `heap = [(0, origem)]`. A cada iteração, extraia o par `(d_u, u)` com `heapq.heappop`. Se `d_u > dist[u]`, a entrada está desatualizada e pode ser descartada. Caso contrário, relaxe os arcos de saída de `u` segundo {eq}`eq-relaxacao`.
:::